In [73]:
from pyspark import SparkContext

In [74]:
sc = SparkContext(master='local', appName='transformacionesYAcciones')

In [75]:
rdd = sc.parallelize([1,2,3])
type(rdd)

pyspark.core.rdd.RDD

In [76]:
rdd.collect()

[1, 2, 3]

In [77]:
!ls /home/jovyan/data/

deporte.csv	 deportistaError.csv  modelo_relacional.jpg
deportista2.csv  evento.csv	      paises.csv
deportista.csv	 juegos.csv	      resultados.csv


In [78]:
path = "/home/jovyan/data/"

In [79]:
equiposOlimpicosRDD = sc.textFile(path+"paises.csv") \
.map(lambda line : line.split(","))

In [80]:
equiposOlimpicosRDD.take(15)

[['id', 'equipo', 'sigla'],
 ['1', '30. Februar', 'AUT'],
 ['2', 'A North American Team', 'MEX'],
 ['3', 'Acipactli', 'MEX'],
 ['4', 'Acturus', 'ARG'],
 ['5', 'Afghanistan', 'AFG'],
 ['6', 'Akatonbo', 'IRL'],
 ['7', 'Alain IV', 'SUI'],
 ['8', 'Albania', 'ALB'],
 ['9', 'Alcaid', 'POR'],
 ['10', 'Alcyon-6', 'FRA'],
 ['11', 'Alcyon-7', 'FRA'],
 ['12', 'Aldebaran', 'ITA'],
 ['13', 'Aldebaran II', 'ITA'],
 ['14', 'Aletta', 'IRL']]

In [81]:
equiposOlimpicosRDD.map(lambda x: (x[2])).distinct().count()

231

In [82]:
equiposOlimpicosRDD.map(lambda x: (x[2], x[1])).groupByKey().mapValues(len).take(5)

[('sigla', 1), ('AUT', 11), ('MEX', 9), ('ARG', 18), ('AFG', 1)]

In [83]:
equiposOlimpicosRDD.map(lambda x: (x[2], x[1])).groupByKey() \
    .mapValues(list).take(5)

[('sigla', ['equipo']),
 ('AUT',
  ['30. Februar',
   'Austria',
   'Austria-1',
   'Austria-2',
   'Breslau',
   'Brigantia',
   'Donar III',
   'Evita VI',
   'May-Be 1960',
   '"R.-V. Germania; Leitmeritz"',
   'Surprise']),
 ('MEX',
  ['A North American Team',
   'Acipactli',
   'Chamukina',
   'Mexico',
   'Mexico-1',
   'Mexico-2',
   'Nausikaa 4',
   'Tlaloc',
   'Xolotl']),
 ('ARG',
  ['Acturus',
   'Antares',
   'Arcturus',
   'Ardilla',
   'Argentina',
   'Argentina-1',
   'Argentina-2',
   'Blue Red',
   'Covunco III',
   'Cupidon III',
   'Djinn',
   'Gullvinge',
   'Matrero II',
   'Mizar',
   'Pampero',
   'Rampage',
   'Tango',
   'Wiking']),
 ('AFG', ['Afghanistan'])]

In [84]:
equiposArgentinos = equiposOlimpicosRDD.filter(lambda l : "ARG" in l)
equiposArgentinos.collect()

[['4', 'Acturus', 'ARG'],
 ['37', 'Antares', 'ARG'],
 ['42', 'Arcturus', 'ARG'],
 ['43', 'Ardilla', 'ARG'],
 ['45', 'Argentina', 'ARG'],
 ['46', 'Argentina-1', 'ARG'],
 ['47', 'Argentina-2', 'ARG'],
 ['119', 'Blue Red', 'ARG'],
 ['238', 'Covunco III', 'ARG'],
 ['252', 'Cupidon III', 'ARG'],
 ['288', 'Djinn', 'ARG'],
 ['436', 'Gullvinge', 'ARG'],
 ['644', 'Matrero II', 'ARG'],
 ['672', 'Mizar', 'ARG'],
 ['774', 'Pampero', 'ARG'],
 ['843', 'Rampage', 'ARG'],
 ['1031', 'Tango', 'ARG'],
 ['1162', 'Wiking', 'ARG']]

In [85]:
equiposOlimpicosRDD.countApprox(20)

1185

In [86]:
deportistaOlimpicoRDD = sc.textFile(path+"deportista.csv") \
    .map(lambda l : l.split(","))
deportistaOlimpicoRDD2 = sc.textFile(path+"deportista2.csv") \
    .map(lambda l : l.split(","))

In [87]:
deportistaOlimpicoRDD = deportistaOlimpicoRDD \
    .union(deportistaOlimpicoRDD2)

In [88]:
deportistaOlimpicoRDD.count()

135572

In [89]:
deportistaOlimpicoRDD.top(2)

[['deportista_id', 'nombre', 'genero', 'edad', 'altura', 'peso', 'equipo_id'],
 ['99999', 'Alexander Grant Alick Rennie', '1', '32', '182', '71', '967']]

In [90]:
deportistaOlimpicoRDD.map(lambda l: [l[6], l[:6]]) \
.join(equiposOlimpicosRDD.map(lambda x: [x[0], x[2]])) \
.takeSample(False,5,25)

[('970', (['68062', 'Lee MinHui', '2', '28', '174', '65'], 'KOR')),
 ('154', (['39161', 'Angel Merdzhanov Gavrilov', '1', '24', '0', '0'], 'BUL')),
 ('1084',
  (['62843', 'Olha Vasylivna Korobka', '2', '18', '181', '167'], 'UKR')),
 ('678', (['97550', 'Puntsagiin Skhbat', '1', '24', '174', '82'], 'MGL')),
 ('1096', (['106789', 'Hugo Scherzer', '1', '43', '0', '0'], 'USA'))]

In [91]:
resultado = sc.textFile(path+"resultados.csv") \
.map(lambda l : l.split(","))

In [92]:
resultadoGanador = resultado.filter(lambda l : 'NA' not in l[1])

In [93]:
resultadoGanador.take(2)

[['resultado_id', 'medalla', 'deportista_id', 'juego_id', 'evento_id'],
 ['4', 'Gold', '4', '2', '4']]

In [94]:
deportistaPais = deportistaOlimpicoRDD \
.map(lambda l :[l[-1], l[:-1]]) \
.join(equiposOlimpicosRDD.map(lambda x :[x[0], x[2]]))

In [95]:
deportistaPais.take(6)

[('199', (['1', 'A Dijiang', '1', '24', '180', '80'], 'CHN')),
 ('199', (['2', 'A Lamusi', '1', '23', '170', '60'], 'CHN')),
 ('199', (['602', 'Abudoureheman', '1', '22', '182', '75'], 'CHN')),
 ('199', (['1463', 'Ai Linuer', '1', '25', '160', '62'], 'CHN')),
 ('199', (['1464', 'Ai Yanhan', '2', '14', '168', '54'], 'CHN')),
 ('199', (['3605', 'An Weijiang', '1', '22', '178', '72'], 'CHN'))]

In [96]:
deportistaPais.map(lambda x: (x[1][0][0], x)).join(resultadoGanador.map(lambda y: (y[2], y[1]))).take(5)

[('7597',
  (('199', (['7597', 'Bao Yingying', '2', '24', '172', '67'], 'CHN')),
   'Silver')),
 ('17282',
  (('199', (['17282', 'Cai Huijue', '2', '16', '174', '63'], 'CHN')),
   'Bronze')),
 ('17996',
  (('199', (['17996', 'Cao Mianying', '2', '21', '176', '71'], 'CHN')),
   'Silver')),
 ('19779',
  (('199', (['19779', 'Chang Si', '2', '25', '170', '56'], 'CHN')), 'Silver')),
 ('19791',
  (('199', (['19791', 'Chang Yongxiang', '1', '24', '178', '74'], 'CHN')),
   'Silver'))]

In [97]:
valoresMedallas = {'Gold' : 7, 'Silver' : 5, 'Bronze' : 4}

In [98]:
paisesMedallas = deportistaPais.join(resultadoGanador)

In [99]:
paisesMedallas.map(lambda x: (x[1][0][-1], valoresMedallas[x[1][1]]))

PythonRDD[67] at RDD at PythonRDD.scala:56

In [100]:
from operator import add
rdd_total = paisesMedallas.map(lambda x: (x[1][0][-1], valoresMedallas[x[1][1]]))
respuesta = rdd_total.reduceByKey(add) \
.sortBy(lambda x :x[1], ascending=False)

In [101]:
respuesta.take(5)

[('CAN', 32538), ('ARG', 12520), ('HUN', 10860), ('MEX', 6124), ('RSA', 3788)]

In [102]:
def eliminaEncabezados(indice, iterador):
    return iter(list(iterador)[1:])

In [103]:
deportistaOlimpicoRDD = deportistaOlimpicoRDD.mapPartitionsWithIndex(eliminaEncabezados)

In [104]:
deportistaOlimpicoRDD.take(5)

[['1', 'A Dijiang', '1', '24', '180', '80', '199'],
 ['2', 'A Lamusi', '1', '23', '170', '60', '199'],
 ['3', 'Gunnar Nielsen Aaby', '1', '24', '0', '0', '273'],
 ['4', 'Edgar Lindenau Aabye', '1', '34', '0', '0', '278'],
 ['5', 'Christine Jacoba Aaftink', '2', '21', '185', '82', '705']]

In [105]:
deportistaOlimpicoRDD = deportistaOlimpicoRDD.map(lambda l : (
    int(l[0]),
    l[1],
    int(l[2]),
    int(l[3]),
    int(l[4]),
    float(l[5]),
    int(l[6])
))

In [106]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
from pyspark.sql.types import Row
from pyspark.sql import SQLContext

In [107]:
sqlContext = SQLContext(sc)

In [142]:
schema = StructType([
StructField("deportista_id",IntegerType(),False),
StructField("nombre",StringType(),False),
StructField("genero",IntegerType(),False),
StructField("edad",IntegerType(),True),
StructField("altura",IntegerType(),True),
StructField("peso",FloatType(),True),
StructField("equipo_id",IntegerType(),True)     
])

In [143]:
deportistaOlimpicoDF = sqlContext.createDataFrame(deportistaOlimpicoRDD,schema)

In [144]:
deportistaOlimpicoDF.printSchema()

root
 |-- deportista_id: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- genero: integer (nullable = false)
 |-- edad: integer (nullable = true)
 |-- altura: integer (nullable = true)
 |-- peso: float (nullable = true)
 |-- equipo_id: integer (nullable = true)



In [148]:
paisesRDD = sc.textFile(path+"paises.csv").map(lambda line : line.split(","))
paisesRDD = paisesRDD.mapPartitionsWithIndex(eliminaEncabezados)

paisesRDD = paisesRDD.map(lambda l : (
int(l[0]),
l[1],
l[2]
))

schema = StructType([
StructField("id",IntegerType(),False),
StructField("equipo",StringType(),False),
StructField("sigla",StringType(),False)
])

paisesDF = sqlContext.createDataFrame(paisesRDD,schema)

In [149]:
eventoSchema= StructType([
    StructField("evento_id",IntegerType(),False),
    StructField("nombre",StringType(),False),
    StructField("deporte_id",IntegerType(),False)
])

deportesOlimpicosDF = sqlContext.read.schema(eventoSchema).option("header","true").csv(path+"evento.csv")

In [150]:
juegoSchema = StructType([
    StructField("juego_id",IntegerType(),False),
    StructField("anio",StringType(),False),
    StructField("temporada",StringType(),False),
    StructField("ciudad",StringType(),False),
])
juegoDF = sqlContext.read.schema(juegoSchema).option("header","true").csv(path+"juegos.csv")

resultadoSchema = StructType([
    StructField("resultado_id",IntegerType(),False),
    StructField("medalla",StringType(),False),
    StructField("deportista_id",IntegerType(),False),
    StructField("juego_id",IntegerType(),False),
    StructField("evento_id",IntegerType(),False),
])
resultadoDF = sqlContext.read.schema(resultadoSchema).option("header","true").csv(path+"resultados.csv")

In [151]:
deportesDF.take(5)

[Row(deporte_id=1, deporte='Basketball'),
 Row(deporte_id=2, deporte='Judo'),
 Row(deporte_id=3, deporte='Football'),
 Row(deporte_id=4, deporte='Tug-Of-War'),
 Row(deporte_id=5, deporte='Speed Skating')]

In [153]:
deportesOlimpicosDF.take(5)

25/12/04 15:02:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: evento_id, evento, deporte_id
 Schema: evento_id, nombre, deporte_id
Expected: nombre but found: evento
CSV file: file:///home/jovyan/data/evento.csv


[Row(evento_id=1, nombre="Basketball Men's Basketball", deporte_id=1),
 Row(evento_id=2, nombre="Judo Men's Extra-Lightweight", deporte_id=2),
 Row(evento_id=3, nombre="Football Men's Football", deporte_id=3),
 Row(evento_id=4, nombre="Tug-Of-War Men's Tug-Of-War", deporte_id=4),
 Row(evento_id=5, nombre="Speed Skating Women's 500 metres", deporte_id=5)]

In [154]:
paisesDF.take(5)

[Row(id=1, equipo='30. Februar', sigla='AUT'),
 Row(id=2, equipo='A North American Team', sigla='MEX'),
 Row(id=3, equipo='Acipactli', sigla='MEX'),
 Row(id=4, equipo='Acturus', sigla='ARG'),
 Row(id=5, equipo='Afghanistan', sigla='AFG')]

In [155]:
juegoDF.take(5)

25/12/04 15:03:14 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 5, schema size: 4
CSV file: file:///home/jovyan/data/juegos.csv


[Row(juego_id=1, anio='1896 Verano', temporada='1896', ciudad='Verano'),
 Row(juego_id=2, anio='1900 Verano', temporada='1900', ciudad='Verano'),
 Row(juego_id=3, anio='1904 Verano', temporada='1904', ciudad='Verano'),
 Row(juego_id=4, anio='1906 Verano', temporada='1906', ciudad='Verano'),
 Row(juego_id=5, anio='1908 Verano', temporada='1908', ciudad='Verano')]

In [156]:
deportistaOlimpicoDF.take(5)

[Row(deportista_id=1, nombre='A Dijiang', genero=1, edad=24, altura=180, peso=80.0, equipo_id=199),
 Row(deportista_id=2, nombre='A Lamusi', genero=1, edad=23, altura=170, peso=60.0, equipo_id=199),
 Row(deportista_id=3, nombre='Gunnar Nielsen Aaby', genero=1, edad=24, altura=0, peso=0.0, equipo_id=273),
 Row(deportista_id=4, nombre='Edgar Lindenau Aabye', genero=1, edad=34, altura=0, peso=0.0, equipo_id=278),
 Row(deportista_id=5, nombre='Christine Jacoba Aaftink', genero=2, edad=21, altura=185, peso=82.0, equipo_id=705)]

In [157]:
resultadoDF.take(5)

[Row(resultado_id=1, medalla='NA', deportista_id=1, juego_id=39, evento_id=1),
 Row(resultado_id=2, medalla='NA', deportista_id=2, juego_id=49, evento_id=2),
 Row(resultado_id=3, medalla='NA', deportista_id=3, juego_id=7, evento_id=3),
 Row(resultado_id=4, medalla='Gold', deportista_id=4, juego_id=2, evento_id=4),
 Row(resultado_id=5, medalla='NA', deportista_id=5, juego_id=36, evento_id=5)]

In [158]:
deportesDF.printSchema()

root
 |-- deporte_id: integer (nullable = true)
 |-- deporte: string (nullable = true)



In [159]:
deportistaOlimpicoDF.printSchema()

root
 |-- deportista_id: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- genero: integer (nullable = false)
 |-- edad: integer (nullable = true)
 |-- altura: integer (nullable = true)
 |-- peso: float (nullable = true)
 |-- equipo_id: integer (nullable = true)



In [160]:
deportistaOlimpicoDF = deportistaOlimpicoDF.withColumnRenamed("genero","sexo").drop("altura")

In [161]:
deportistaOlimpicoDF.printSchema()

root
 |-- deportista_id: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- sexo: integer (nullable = false)
 |-- edad: integer (nullable = true)
 |-- peso: float (nullable = true)
 |-- equipo_id: integer (nullable = true)



In [162]:
from pyspark.sql.functions import *
deportistaOlimpicoDF = deportistaOlimpicoDF.select("deportista_id","nombre",
                            col("edad").alias("edadAlJugar"),"equipo_id")

In [163]:
deportistaOlimpicoDF.show(5)

+-------------+--------------------+-----------+---------+
|deportista_id|              nombre|edadAlJugar|equipo_id|
+-------------+--------------------+-----------+---------+
|            1|           A Dijiang|         24|      199|
|            2|            A Lamusi|         23|      199|
|            3| Gunnar Nielsen Aaby|         24|      273|
|            4|Edgar Lindenau Aabye|         34|      278|
|            5|Christine Jacoba ...|         21|      705|
+-------------+--------------------+-----------+---------+
only showing top 5 rows


In [164]:
deportistaOlimpicoDF = deportistaOlimpicoDF.filter( (deportistaOlimpicoDF.edadAlJugar != 0))

In [165]:
deportistaOlimpicoDF.sort("edadAlJugar").show()

[Stage 51:=============================>                            (1 + 1) / 2]

+-------------+--------------------+-----------+---------+
|deportista_id|              nombre|edadAlJugar|equipo_id|
+-------------+--------------------+-----------+---------+
|        71691|  Dimitrios Loundras|         10|      333|
|        22411|Magdalena Cecilia...|         11|      413|
|        70616|          Liu Luyang|         11|      199|
|        37333|Carlos Bienvenido...|         11|      982|
|        76675|   Marcelle Matthews|         11|      967|
|        40129|    Luigina Giavotti|         11|      507|
|       118925|Megan Olwen Deven...|         11|      413|
|        47618|Sonja Henie Toppi...|         11|      742|
|       126307|        Liana Vicens|         11|      825|
|        51268|      Beatrice Hutiu|         11|      861|
|        52070|        Etsuko Inada|         11|      514|
|        72854|      Licia Macchini|         12|      507|
|         5291|Marcia Arriaga La...|         12|      656|
|        74712|     Carla Marangoni|         12|      50

In [168]:
deportistaOlimpicoDF.join(resultadoDF, deportistaOlimpicoDF.deportista_id == resultadoDF.deportista_id,"left") \
    .join(juegoDF,juegoDF.juego_id == resultadoDF.juego_id,"left") \
    .join(deportesOlimpicosDF, deportesOlimpicosDF.evento_id == resultadoDF.evento_id,"left") \
    .select(deportistaOlimpicoDF.nombre, col("edad").alias("Edad el jugar"),
           "medalla",col("anio").alias("Año de juego"),
           deportesOlimpicosDF.nombre.alias("Nombre de disciplina")).show()

{"ts": "2025-12-04 15:06:34.176", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `edad` cannot be resolved. Did you mean one of the following? [`ciudad`, `anio`, `medalla`, `juego_id`, `juego_id`]. SQLSTATE: 42703", "context": {"file": "line 4 in cell [168]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1465.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `edad` cannot be resolved. Did you mean one of the following? [`ciudad`, `anio`, `medalla`, `juego_id`, `juego_id`]. SQLSTATE: 42703;\n'Project [nombre#268, 'edad AS Edad el jugar#342, medalla#287, anio#283 AS Año de juego#343, nombre#280 AS Nombre de disciplina#344]\n+- Join LeftOuter, (evento_id#279 = evento_id#290)\n   

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `edad` cannot be resolved. Did you mean one of the following? [`ciudad`, `anio`, `medalla`, `juego_id`, `juego_id`]. SQLSTATE: 42703;
'Project [nombre#268, 'edad AS Edad el jugar#342, medalla#287, anio#283 AS Año de juego#343, nombre#280 AS Nombre de disciplina#344]
+- Join LeftOuter, (evento_id#279 = evento_id#290)
   :- Join LeftOuter, (juego_id#282 = juego_id#289)
   :  :- Join LeftOuter, (deportista_id#267 = deportista_id#288)
   :  :  :- Filter NOT (edadAlJugar#309 = 0)
   :  :  :  +- Project [deportista_id#267, nombre#268, edad#270 AS edadAlJugar#309, equipo_id#273]
   :  :  :     +- Project [deportista_id#267, nombre#268, sexo#308, edad#270, peso#272, equipo_id#273]
   :  :  :        +- Project [deportista_id#267, nombre#268, genero#269 AS sexo#308, edad#270, altura#271, peso#272, equipo_id#273]
   :  :  :           +- LogicalRDD [deportista_id#267, nombre#268, genero#269, 